# Эксперимент Дениса (dense)
Повторяем обычный (dense) запуск из ноутбука Дениса, используя только модули transformer_lab.

Отличия от Дениса:
- токенизатор я взял готовый GPT-2 BPE (50 257) - мы тестируем модель, а не токенизатор, поэтому можно взять готовый;
- корпус — один поток без `eos` между статьями - опять же, чтобы не усложнять систему. Мы тестируем не это, поэтому можно просто взять что-то готовое;
- убрал лимит на количество токенов, которые участвуют в обучении

Устанавливаем наш пакет

In [ ]:
%pip install -q git+https://github.com/tereshkovyv/transformer_lab.git

Подключаем Гугл диск, чтобы хранить там временные файл

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/transformer_lab/data/wikitext-103"
RUN_DIR = "/content/drive/MyDrive/transformer_lab/runs/denis_dense_seed44"

Настраиваем конфигурацию

In [ ]:
SEED = 44
cfg = {"vocab_size": 50257, "context_length": 2048, "emb_dim": 512, "n_heads": 8,
       "n_layers": 12, "hidden_dim": 1536, "drop_rate": 0.0, "qkv_bias": False}
hp = {"max_length": 512, "batch_size": 8, "lr": 3e-4, "weight_decay": 0.1, "betas": (0.9, 0.95),
      "warmup_steps": 102, "min_lr": 3e-5, "train_steps": 2048, "eval_batch_size": 2,
      "log_freq": 50, "eval_freq": 250, "eval_iter": 8, "checkpoint_freq": 500, "use_amp": True}

Создаем объекты данных

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from transformer_lab.training.data import TokenDataset

device = "cuda"
torch.manual_seed(SEED)

train_ds = TokenDataset("train", hp["max_length"], DATA_DIR)
val_ds = TokenDataset("validation", hp["max_length"], DATA_DIR)

g = torch.Generator().manual_seed(SEED)
train_idx = torch.randperm(len(train_ds), generator=g)[: hp["train_steps"] * hp["batch_size"]]
train_loader = DataLoader(Subset(train_ds, train_idx.tolist()), batch_size=hp["batch_size"],
                          shuffle=False, drop_last=True)  # order is already random and fixed, so a run can resume
val_loader = DataLoader(val_ds, batch_size=hp["eval_batch_size"], shuffle=False, drop_last=False)
print("train steps:", len(train_loader), "val batches:", len(val_loader))

Создаем модель и оптимайзер

In [ ]:
from transformer_lab.model.gpt_model import GPTModelV2

model = GPTModelV2(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"],
                              weight_decay=hp["weight_decay"], betas=hp["betas"])

Обучаем

In [ ]:
import os
import time
import tiktoken
from transformer_lab.training.train import train_model

tokenizer = tiktoken.get_encoding("gpt2")
os.makedirs(RUN_DIR, exist_ok=True)
progress_path = f"{RUN_DIR}/progress.pth"
start = time.perf_counter()
history, lrs = train_model(
    model, train_loader, val_loader, optimizer, device,
    n_epochs=1, eval_freq=hp["eval_freq"], eval_iter=hp["eval_iter"], log_freq=hp["log_freq"],
    start_context="The", tokenizer=tokenizer,
    warmup_steps=hp["warmup_steps"], initial_lr=hp["lr"] / hp["warmup_steps"],
    min_lr=hp["min_lr"], use_amp=hp["use_amp"],
    checkpoint_path=progress_path, checkpoint_freq=hp["checkpoint_freq"])
train_seconds = time.perf_counter() - start
print(f"training: {train_seconds / 60:.1f} min")

Считаем loss и perplexity на всём validation, генерируем образец 30 токенов от "The"

In [ ]:
import math
from transformer_lab.model.generate import generate_text_simple
from transformer_lab.training.train import calc_loss_loader

model.eval()
with torch.no_grad():
    val_loss = calc_loss_loader(val_loader, model, device)
print(f"val loss {val_loss:.4f}, perplexity {math.exp(val_loss):.2f}")

encoded = torch.tensor(tokenizer.encode("The")).unsqueeze(0).to(device)
sample = tokenizer.decode(generate_text_simple(model, encoded, max_new_tokens=30,
                                               context_size=hp["max_length"]).squeeze(0).tolist())
print(sample)

Рисуем кривые обучения

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
val_points = [(t, v) for t, v in zip(history["tokens_seen"], history["val_loss"]) if v is not None]
axes[0].plot(history["tokens_seen"], history["train_loss"], label="train (current step)")
axes[0].plot(*zip(*val_points), marker="o", markersize=3, label="validation (16 sequences)")
axes[0].set(title="Loss", xlabel="Processed tokens", ylabel="Loss")
axes[0].legend()
axes[1].plot(lrs)
axes[1].set(title="Learning rate", xlabel="Step", ylabel="lr")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

Сохраняем веса, после этого удаляем файл прогресса с Google Drive

In [ ]:
from google.colab import files
torch.save(model.state_dict(), "model.pth")
files.download("model.pth")

# Only after the weights are saved: if Colab died before this point, the next run would still resume from it
if os.path.exists(progress_path):
    os.remove(progress_path)